# REM_Turku dream-affect decoding: baselines, nulls, and what they sayHandoff for **Paul Barbaste**. Runs top to bottom on Colab (GPU optional; the tangent andTSMNet arms are fine on CPU). No cluster paths, no credentials.**Read this first.** Every arm below is null. That is the result, not a work-in-progress.What this notebook gives you is a harness where a new method is ~20 lines, scored onexactly the folds every existing arm used, against a permutation null built the same way.Three things that cost us days and are already fixed here:1. Permutation nulls shuffle labels **within subject**. A global shuffle redraws each   subject's class prior toward the grand mean, which biases the null rather than widening   it. It inflated our detectable-effect floor from 0.598 to 0.655 and manufactured a   fold-dependence that does not exist.2. `seed 0` means **no shuffle**. A null loop starting at 0 puts the unshuffled observed   inside its own null.3. A permutation p sitting at its floor (zero exceedances) is **not** a measurement. Ours   read 0.0175 at 56 draws and 0.0550 at 199.

## 0. Configuration`DATA_DIR` is the only path you should need to touch.

In [ ]:
import os, sys, subprocess, hashlib, json, zipfile, io, csvfrom pathlib import PathDATA_DIR = Path(os.environ.get("REMTURKU_DIR", "./remturku_data"))DATA_DIR.mkdir(parents=True, exist_ok=True)# The deposit is public: Sikka, Revonsuo, Noreika & Valli, figshare# 10.6084/m9.figshare.23274596.v2 . 18 subjects deposited / 17 usable, 2 nights each,# 133 REM awakenings with a self-rated 20-item mDES, 24 EEG + 4 EOG + 1 EMG at 500 Hz.FIGSHARE_DOI = "10.6084/m9.figshare.23274596.v2"ZIP_PATH = DATA_DIR / "REM_Turku.zip"NPZ_PATH = DATA_DIR / "remturku_epochs.npz"print("DATA_DIR:", DATA_DIR.resolve())print("zip present:", ZIP_PATH.exists(), "| npz present:", NPZ_PATH.exists())

In [ ]:
!pip -q install pyriemann==0.12 braindecode mne scikit-learn scipy 2>&1 | tail -2# TSMNet arm only; skip if you are not running it# !pip -q install git+https://github.com/rkobler/TSMNet

## 1. Data**We deliberately do not ship a preprocessed archive.** The prepared `.npz` is 2.8 GB, itwould dominate a Colab session, and shipping it would mean you trust our artefact insteadof verifying your environment. The cell below downloads the raw deposit and rebuilds theepochs with the same script and the same pipeline version string(`remturku-v1-sikka2019`) that produced every number in this notebook.First run is ~15-20 minutes. After that it is cached in `DATA_DIR`.Preprocessing, fixed and matching Sikka et al. 2019's descriptives: 24 EEG channels(10/10), 0.5-45 Hz, average reference, ICA with EOG components excluded (1.78 per file onaverage), 2 s epochs with 0.5 overlap, 200 uV rejection, 99.7% of epochs kept. Both raw andCSD versions are written; **every arm here uses `raw`** because CSD is a spatial high-passthat removes what the covariance and the CNN both need.

In [ ]:
def fetch_deposit():    if ZIP_PATH.exists():        print("zip already present:", ZIP_PATH); return    url = f"https://api.figshare.com/v2/articles/23274596/files"    print("resolving figshare files for", FIGSHARE_DOI)    import urllib.request    meta = json.loads(urllib.request.urlopen(url).read())    target = max(meta, key=lambda f: f["size"])          # the deposit archive    print(f"downloading {target['name']} ({target['size']/1e9:.2f} GB)")    urllib.request.urlretrieve(target["download_url"], ZIP_PATH)    print("done:", ZIP_PATH)def build_epochs():    if NPZ_PATH.exists():        print("npz already present:", NPZ_PATH); return    # prepare_remturku.py lives in this repo under data/preprocessing/    script = Path("prepare_remturku.py")    if not script.exists():        raise FileNotFoundError(            "prepare_remturku.py not found. Clone the repo first:\n"            "  !git clone https://github.com/hollanderski/readream-riemann-baseline\n"            "  %cd readream-riemann-baseline")    subprocess.run([sys.executable, str(script), "--zip", str(ZIP_PATH),                    "--out", str(NPZ_PATH), "--save-epochs"], check=True)# fetch_deposit(); build_epochs()      # <- uncomment to run the ~20 min rebuildprint("data cell defined; uncomment the last line to fetch and build")

## 2. The harnessOne loader, one fold order, one scorer. Every arm in section 3 is a `fit_predict` callableand gets the identical treatment. **This is the part worth reusing**: it is what makes anew method comparable to the existing ones rather than merely adjacent to them.`HVDC` maps mDES items to Hall & Van de Castle categories by presence (`SR > 0`). Note`confusion` maps from `SR_PA2` (Awe/Wonder/Amazement), a **positive** item mapped to anon-positive HVdC category. That mapping is inherited from the project spec and isvalence-inconsistent; it is flagged, not silently changed.

In [ ]:
import numpy as np, statistics as stfrom scipy.stats import wilcoxonHVDC = {"anger":        ["SR_NA1", "SR_NA7"],    # Angry/Irritated, Hate/Distrust        "apprehension": ["SR_NA9", "SR_NA10"],   # Scared/Fearful, Stressed/Nervous        "confusion":    ["SR_PA2"]}              # Awe/Wonder  <- see the caveat abovedef load(target, epoch_kind="raw"):    """-> (list of (n_epochs, 24, 1000) arrays, y per awakening, subject per awakening)"""    z = zipfile.ZipFile(ZIP_PATH)    rd = lambda p: csv.DictReader(io.StringIO(z.read(p).decode("utf-8-sig")))    rat = {r["Filename"]: r for r in rd("REM_Turku/Data/Ratings.csv")}    rec = {r["Filename"]: r for r in rd("REM_Turku/Records.csv")}    npz = np.load(NPZ_PATH)    f = lambda v: (float(v) if str(v).replace(".", "", 1).isdigit() else None)    X, y, s = [], [], []    for fn, r in rat.items():        k = f"{fn}|{epoch_kind}"        if k not in npz or fn not in rec: continue        if not any((f(r[c]) or 0) > 0 for c in r if c.startswith("SR_")): continue        X.append(npz[k]); y.append(int(any((f(r[c]) or 0) > 0 for c in HVDC[target])))        s.append(rec[fn]["Subject ID"])    return X, np.array(y), np.array(s)def bal_acc(pred, true):    tp = ((pred==1)&(true==1)).sum(); fn = ((pred==0)&(true==1)).sum()    tn = ((pred==0)&(true==0)).sum(); fp = ((pred==1)&(true==0)).sum()    se = tp/(tp+fn) if tp+fn else 0.0    sp = tn/(tn+fp) if tn+fp else 0.0    return float((se+sp)/2)def within_subject_shuffle(y, s, seed):    """Permutation null. Each subject keeps its OWN class prior.    A global permutation redraws every base rate toward the grand mean, changes the    between-subject prior structure the LOSO statistic is computed over, and changes which    folds come out degenerate. That is a biased null, not a wide one.    seed 0 is refused: it means no shuffle, and a null loop starting there puts the    unshuffled observed inside its own null."""    assert seed >= 1, "seed 0 means NO shuffle and must never be a null draw"    y = np.asarray(y).copy(); rng = np.random.default_rng(seed)    for u in np.unique(s):        m = s == u; y[m] = rng.permutation(y[m])    return y

In [ ]:
def evaluate(fit_predict, target, shuffle_seed=0, epoch_kind="raw", sub=2, verbose=True):    """Leave-one-subject-out over all 17 subjects. THE entry point.    fit_predict(Xtr, ytr, Xte) -> predictions for Xte, where X is (n_epochs, 24, n_times)    and every epoch of a held-out subject is one row of Xte.    Returns {subject: balanced accuracy}. Subjects whose held-out set is single-class are    scored None and excluded, which is why n differs between targets (14 / 16 / 13).    Scores are EPOCH level; see section 5 for why that matters."""    X, y, s = load(target, epoch_kind)    if shuffle_seed: y = within_subject_shuffle(y, s, shuffle_seed)    out = {}    for held in sorted(set(s), key=int):        te = s == held; tr = ~te        if y[te].sum() in (0, te.sum()):            out[held] = None; continue        win = lambda m: (np.concatenate([X[i][::sub] for i in np.where(m)[0]]) * 1e6,                         np.concatenate([[y[i]]*len(X[i][::sub]) for i in np.where(m)[0]]))        Xtr, ytr = win(tr); Xte, yte = win(te)        pred = fit_predict(Xtr, ytr, Xte)        out[held] = bal_acc(np.asarray(pred), yte)        if verbose: print(f"  subj {held}: {out[held]:.4f}")    v = [x for x in out.values() if x is not None]    if verbose: print(f"{target}: mean {np.mean(v):.4f} over {len(v)} subjects")    return outdef permutation_null(fit_predict, target, n_draws=100, **kw):    """n_draws within-subject shuffles, seeds 1..n_draws. Reports the EMPIRICAL 95th    percentile alongside p, and says so when p is floor-limited."""    obs = evaluate(fit_predict, target, verbose=False, **kw)    o = np.mean([x for x in obs.values() if x is not None])    draws = []    for seed in range(1, n_draws+1):           # starts at 1, never 0        d = evaluate(fit_predict, target, shuffle_seed=seed, verbose=False, **kw)        draws.append(np.mean([x for x in d.values() if x is not None]))        if seed % 10 == 0: print(f"  {seed}/{n_draws} draws")    ge = sum(1 for x in draws if x >= o); p = (ge+1)/(len(draws)+1)    q95 = float(np.percentile(draws, 95))    print(f"observed {o:.4f} | null mean {np.mean(draws):.4f} sd {np.std(draws):.4f}")    print(f"empirical 95th pct {q95:.4f} | exceeded {ge}/{len(draws)} -> p = {p:.4f}"          + ("   AT FLOOR, NOT RESOLVED" if ge == 0 else "   resolved"))    return {"observed": float(o), "p": float(p), "n_draws": len(draws),            "null_mean": float(np.mean(draws)), "q95": q95, "floor_limited": ge == 0}

## 3. The armsEach is a factory returning a `fit_predict`. Add yours the same way and section 4 willscore it against every existing arm on identical folds.

In [ ]:
from pyriemann.estimation import Covariancesfrom pyriemann.tangentspace import TangentSpacefrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDAfrom sklearn.linear_model import LogisticRegressiontry:    from pyriemann.geometry.mean import mean_riemannexcept ImportError: from pyriemann.utils.mean import mean_riemann   # <0.14def tangent_global(clf="lda"):    """Rung 1: ONE Frechet reference over the pooled training covariances, applied to the    held-out subject as well. No domain adaptation."""    def fp(Xtr, ytr, Xte):        cov = Covariances(estimator="oas")        Ctr, Cte = cov.fit_transform(Xtr), cov.transform(Xte)        ts = TangentSpace(metric="riemann")        Ttr, Tte = ts.fit_transform(Ctr), ts.transform(Cte)        m = (LDA(solver="lsqr", shrinkage="auto") if clf == "lda"             else LogisticRegression(max_iter=1000)).fit(Ttr, ytr)        return m.predict(Tte)    return fpdef tangent_recentred(clf="lda"):    """Rung 2: Zanini recentring. Each subject is whitened by its OWN Riemannian mean.    The held-out subject uses its own UNLABELLED mean, which is what makes this legal under    LOSO: no test label enters. It is transductive (the whole test set at once), standard    for unsupervised RPA, and stated rather than hidden.    On this corpus it LOSES to rung 1 on all three targets, threshold-free as well as at a    threshold. Recentring removes subject MEANS; see section 5 for why that is the wrong    operation here."""    def recentre(C):        M = mean_riemann(C); w, V = np.linalg.eigh(M)        Mi = V @ np.diag(1.0/np.sqrt(np.maximum(w, 1e-12))) @ V.T        return Mi @ C @ Mi    def fp(Xtr, ytr, Xte):        cov = Covariances(estimator="oas")        Ctr, Cte = recentre(cov.fit_transform(Xtr)), recentre(cov.transform(Xte))        ts = TangentSpace(metric="riemann")        Ttr, Tte = ts.fit_transform(Ctr), ts.transform(Cte)        m = (LDA(solver="lsqr", shrinkage="auto") if clf == "lda"             else LogisticRegression(max_iter=1000)).fit(Ttr, ytr)        return m.predict(Tte)    return fp

In [ ]:
import torch, torch.nn as nnfrom braindecode.models import ShallowFBCSPNetdef braindecode_shallow(epochs=100, lr=3e-4, batch=64, device=None):    """braindecode ShallowFBCSPNet at LIBRARY DEFAULTS. 42,682 parameters.    Defaults are deliberate: a 14-config sweep reproduced them to within 0.0006, and    nested selection across five arms changed nothing (section 5). Note the library    defaults filter_time_length=25 / pool_time_length=75 are calibrated for 250 Hz; this    corpus is 500 Hz, so they are half the intended physiological scales."""    device = device or ("cuda" if torch.cuda.is_available() else "cpu")    def fp(Xtr, ytr, Xte):        # volts -> microvolts happens in evaluate(); z-score on TRAINING stats only        mu, sd = Xtr.mean(axis=(0,2), keepdims=True), Xtr.std(axis=(0,2), keepdims=True)+1e-8        Xtr, Xte = (Xtr-mu)/sd, (Xte-mu)/sd        net = ShallowFBCSPNet(Xtr.shape[1], 2, n_times=Xtr.shape[2],                              final_conv_length="auto").to(device)        opt = torch.optim.AdamW(net.parameters(), lr=lr)        lossf = nn.CrossEntropyLoss()        xt = torch.tensor(Xtr, dtype=torch.float32); yt = torch.tensor(ytr, dtype=torch.long)        net.train()        for ep in range(epochs):            perm = torch.randperm(len(xt))            for i in range(0, len(xt), batch):                j = perm[i:i+batch]                opt.zero_grad()                loss = lossf(net(xt[j].to(device)), yt[j].to(device))                loss.backward(); opt.step()        net.eval()        with torch.no_grad():            out = net(torch.tensor(Xte, dtype=torch.float32).to(device))        return out.argmax(1).cpu().numpy()    return fp

### TSMNet / SPDNetInstall with `pip install git+https://github.com/rkobler/TSMNet` and use the repo**unmodified**.**One rule that will cost you an afternoon otherwise.** TSMNet routes its SPD/tangentstage through `spd_device_` in double precision and ships a `CPUModel` mixin, so **themodel's logits are legitimately on CPU even when you called `.to("cuda")`**. Bring theTARGET to the logits, never the model to the device:```pythonloss = lossf(logits, y.to(logits.device))```Do **not** walk the modules forcing every stray tensor to cuda. It makes the crash goaway, moves the SPD stage off its intended device, and you get numbers from anarchitecture you silently altered. (There is also a real bug at `spdnets/batchnorm.py:206`where `self.mean` is a bare tensor rather than a registered buffer, so `.to()` skips it,but the device rule above is the correct fix.)

## 4. Results, frozen 2026-08-31**Every arm is null.** Read each number against its own permutation null, never against0.5, and never against another arm's null.### Cross-subject LOSO, 17 subjects, EPOCH-level balanced accuracy| arm | apprehension | anger | confusion ||---|---|---|---|| ShallowFBCSPNet, defaults | 0.5413 (p=0.161) | 0.4748 | 0.4961 || EEGNet, defaults | 0.5507 | - | - || ShallowConv_Embedding, defaults | 0.5347 | - | - || tangent, global reference | **0.5701** (p=0.055, 199 draws) | 0.3904 | 0.4907 || tangent, per-subject recentring | 0.5117 (p=0.398) | 0.3517 | 0.4374 || ShallowFBCSPNet, **nested** 12-config | 0.5571 (n=14) | *paused* | 0.4911 (n=13) || EEGNet, **nested** 12-config | 0.5303 (n=14) | 0.4823 (n=16) | 0.5419 (n=13) |TSMNet defaults reports both levels: apprehension epoch **0.5834** / awakening **0.5852**(n=14, awakening-level null **p=0.1429**, resolved), anger 0.4354/0.4279, confusion0.4753/0.5125.### Within-subject, leave-one-awakening-out, AWAKENING level| target | observed | its own null | p ||---|---|---|---|| apprehension | 0.4477 | 0.3856 (sd 0.066, n=82) | 0.205 || negaff composite | 0.4408 | 0.4241 (sd 0.064, n=52) | 0.434 || anger | 0.3694 | 0.4036 (sd 0.050, n=61) | 0.758 || confusion | 0.3604 | 0.3917 (sd 0.061, n=62) | 0.667 |**None of these is below chance.** Every within-subject null centres between 0.386 and0.424, because at 5-11 training awakenings per fold balanced accuracy does not centre at0.5. Printing them next to 0.5 misleads; we did it twice before catching it.### The one significant result**anger's cross-subject ranking is significantly INVERTED.** Threshold-free AUC 0.3341against a within-subject null centred at 0.5022 over 199 draws, **p = 0.0100**, passesBonferroni over the three labels. A held-out Simpson decomposition gives r = **+0.180BETWEEN** dreamers and r = **-0.236 WITHIN** one: the pooled direction tracks angerpositively across dreamers and negatively inside a dreamer, so a cross-subject decoderlearns the between-subject axis and applies it within the held-out dreamer, where it ranksanger backwards. That also explains why recentring made it worse (0.334 -> 0.318):recentring removes subject *means*, and this axis survives mean removal.Supported, not proven: one label of three, and the mechanism test has no null of its own.

## 5. What the harness will not tell you, and you need to know**Selection signal is below the noise floor at this N.** Measured three independent ways:a 14-config sweep reproduced library defaults to 0.0006; a 6-subject dev split scored all14 configs between 0.426 and 0.515, inside the binomial SE, so its argmax selected noise;and nested selection inside every outer fold changed nothing across five arms and twoarchitectures (deltas -0.020 to +0.046, all inside a fold-to-fold sd of 0.08-0.16), withthe 12 sampled configs spanning 0.388-0.631 on inner validation. **Do not report a sweptnumber here without a nested control.****The pipeline is not the explanation.** Through these exact features, epochs and CV,subject identity decodes at **0.8947** on 17 classes (null mean 0.0576, null max 0.1278over 84 draws) and recording night at **0.8120** (null max 0.5714 over 100). Sex (p=0.089)and circadian position (p=0.088) do not. So the affect nulls are about the labels, in apipeline that demonstrably preserves the strongest signals in EEG.**Detectable-effect floor.** With the correct within-subject null the floor for one-sidedp<0.05 is ~0.598 at n=14. Several of the numbers above sit close enough to it that thedesign could not have detected them even if real. Report the floor next to every accuracy.**Metric level is not cosmetic.** Epochs inside one awakening share its label and arecorrelated, so an epoch-level null is narrower than it should be. The same 20 draws gavep<=0.048 (floor) at epoch level and p=0.1429 (resolved) at awakening level for the sameTSMNet observation. Every row above is labelled; do not mix them.

## 6. Your entry pointWrite a `fit_predict`, pass it to `evaluate`, then to `permutation_null`. That is all.

In [ ]:
# --- worked example: your method here -------------------------------------------------def my_method():    """fit_predict(Xtr, ytr, Xte) -> predictions.    Xtr is (n_epochs, 24, n_times) in microvolts, ytr is 0/1 per epoch."""    from sklearn.pipeline import make_pipeline    def fp(Xtr, ytr, Xte):        cov = Covariances(estimator="oas")        ts  = TangentSpace(metric="riemann")        clf = LDA(solver="lsqr", shrinkage="auto")        Ttr = ts.fit_transform(cov.fit_transform(Xtr))        return clf.fit(Ttr, ytr).predict(ts.transform(cov.transform(Xte)))    return fp# scores = evaluate(my_method(), "apprehension")# result = permutation_null(my_method(), "apprehension", n_draws=100)print("entry point ready")

In [ ]:
def paired_vs(arm_a, arm_b, target, name_a="A", name_b="B"):    """Paired comparison on the SAME subjects. This is the like-for-like quantity: it does    not depend on either arm's null centre, which is why it survived when the individual    p-values did not."""    a = evaluate(arm_a, target, verbose=False); b = evaluate(arm_b, target, verbose=False)    keys = [k for k in a if a[k] is not None and b.get(k) is not None]    d = np.array([a[k]-b[k] for k in keys])    stat, p = wilcoxon(d, alternative="two-sided")    print(f"{name_a} - {name_b} on {target}: {d.mean():+.4f} (sd {d.std():.4f}), "          f"{(d>0).sum()}/{len(d)} positive, Wilcoxon p={p:.4f}")    return {"delta": float(d.mean()), "wins": int((d>0).sum()), "n": len(d), "p": float(p)}# tangent global vs braindecode defaults, apprehension, on our runs:#   +0.0288, 11/14 positive, Wilcoxon p=0.012 one-sided#   while NEITHER arm separates from chance (p=0.055 and p=0.161).# That is the honest headline: a method comparison, not a decoding claim.print("paired cell ready")

## 7. Provenance- Pre-registrations, one per design decision, each committed **before** its numbers  existed: `PREREG.md` (6 addenda + a deviation log).- Every analysis choice that changed after we saw a number is in that deviation log,  including one arm cancelled mid-run on interim grounds and later restarted to completion.- Numbers frozen 2026-08-31. The body_action tuned cells are pending; see notebook 2.- `harness/test_guard.py` asserts every null loop guards its lower bound against seed 0.  Run it before any null.